# Preprocessing & Tokenization

## Load the dataset into a hugging face dataset object

In [ ]:
import datasets


train_set = datasets.load_dataset("csv", data_files="../data/train_set.csv")['train']
train_set

In [ ]:
train_set[:3]['comment_text']

## Preprocessing

In [ ]:
import re

# existing patterns
username_pattern = re.compile(r'\[\[user.*?\]\]|\[\[user talk.*?\]\]|user:\w+', re.IGNORECASE)
whitespace_pattern = re.compile(r'\s+')
url_pattern = re.compile(r'http\S+|www\.\S+')
repeat_char_pattern = re.compile(r'(.)\1{2,}')

# wiki markup patterns
wiki_link_pattern = re.compile(r'\[\[(?:[^\]|]*\|)?([^\]]+)\]\]')
wiki_template_pattern = re.compile(r'\{\{.*?\}\}')
wiki_bold_italic_pattern = re.compile(r"'{2,5}")
wiki_heading_pattern = re.compile(r'=+\s*(.*?)\s*=+')

def preprocess_batch(batch):
    cleaned_texts = []

    for text in batch["comment_text"]:
        text = text.lower()

        # remove usernames
        text = username_pattern.sub(" ", text)

        # replace URLs with a [URL] token
        text = url_pattern.sub("[URL]", text)

        # convert [[link|text]] -> text
        text = wiki_link_pattern.sub(r"\1", text)

        # remove templates {{...}}
        text = wiki_template_pattern.sub(" ", text)

        # remove bold/italic markup
        text = wiki_bold_italic_pattern.sub("", text)

        # remove headings
        text = wiki_heading_pattern.sub(r"\1", text)

        # normalize stretched characters
        text = repeat_char_pattern.sub(r"\1\1", text)

        # normalize whitespace
        text = whitespace_pattern.sub(" ", text).strip()

        cleaned_texts.append(text)

    batch["comment_text"] = cleaned_texts
    return batch

In [ ]:
train_set = train_set.map(preprocess_batch, batched=True, num_proc=4)

In [ ]:
train_set[:3]['comment_text']

Save it to a .csv file

In [ ]:
train_set.to_csv("../data/preproc_train.csv", index=False)

## Tokenization

### Use Pretrained tokenizer: BERT

- It uses WordPiece tokenization algorithm

- Subword Tokenization (WordPiece): Instead of splitting solely by whitespace, WordPiece tokenizes words into subwords based on a pre-trained vocabulary of 30,000 to 30,522 tokens. For instance, "unaffable" might be broken into "un", "##aff", "##able".
Continuation Prefix (##): Subword units that are not the beginning of a word are prefixed with ## to distinguish them as continuations. For example, "playing" might become "play" and "##ing".

#### Special Tokens: 
The tokenizer automatically inserts specific tokens for BERT to understand the structure of the input:

    [CLS]: Placed at the beginning of the first sentence, used for classification tasks.
    [SEP]: Used to separate sentences or to mark the end of a sequence.
    [MASK]: Used during training for masked language modeling.
    [UNK]: Used for characters or words that are not in the vocabulary.
    [PAD]: Used to make input sequences of equal length for batch processing.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
tokenizer

We need to add our custom token for URLs: [URL]

In [ ]:
tokenizer.add_special_tokens(
    {
        "additional_special_tokens": ["[URL]"]
    }
)

In [ ]:
print(tokenizer.convert_tokens_to_ids("[URL]"))

In [ ]:
text = "you are an idiot [URL]"

tokens = tokenizer.tokenize(text)
print(tokens)

In [ ]:
ids = tokenizer(text)
print(ids)

To know what `max_length` to use of the tokenizer, we look at the token **comment length distribution** and choose a percentile. Then we settle around the **95th–99th** percentile so almost all samples fit without wasting memory on extreme outliers.

In [ ]:
def tokenize_for_length(batch):
    tokens = tokenizer(
        batch["comment_text"],
        truncation=False,
        padding=False
    )
    batch["length"] = [len(ids) for ids in tokens["input_ids"]]
    return batch

In [ ]:
dataset_with_length = train_set.map(
    tokenize_for_length,
    batched=True,
    num_proc=4
)

In [ ]:
import numpy as np

lengths = np.array(dataset_with_length["length"])

In [ ]:
np.percentile(lengths, [50, 90, 95, 99])

Verify information loss

In [ ]:
max_length = int(np.percentile(lengths, 95))

truncated_fraction = (lengths > max_length).mean()

print(truncated_fraction)

In [ ]:
import matplotlib.pyplot as plt

plt.hist(lengths, bins=100)
plt.xlabel("Token length")
plt.ylabel("Frequency")
plt.show()

So We will set max_length to 256

In [ ]:
MAX_LENGTH = 256

### Tokenize the dataset

In [ ]:
def tokenize_batch(batch):
    return tokenizer(
        batch["comment_text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

In [ ]:
train_set = train_set.map(
    tokenize_batch,
    batched=True,
    num_proc=4
)

In [ ]:
train_set[10]

Remove unused columns

In [ ]:
train_set = train_set.remove_columns(["comment_text", "id"])

In [ ]:
train_set

Change the format to PyTorch **tensors**. This matters because the PyTorch `DataLoader` expects tensors to batch efficiently.

Otherwise PyTorch must repeatedly convert lists to tensors during training, which wastes time.


In [ ]:
train_set.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "toxic",
        "severe_toxic",
        "obscene",
        "threat",
        "insult",
        "identity_hate"
    ]
)

Save the dataset to a **binary Arrow format**:

In [ ]:
train_set.save_to_disk("../data/tokenized_dataset")

# to load it later:
# from datasets import load_from_disk
# tokenized_dataset = load_from_disk("../data/tokenized_dataset")

Custom collate function

In [ ]:
import torch

LABEL_COLUMNS = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

def collate_fn(batch):
    input_ids = torch.stack([item["input_ids"] for item in batch])
    attention_mask = torch.stack([item["attention_mask"] for item in batch])

    labels = torch.stack([
        torch.tensor([item[label] for label in LABEL_COLUMNS], dtype=torch.float32)
        for item in batch
    ])

    X = {
        "input_ids": input_ids,
        "attention_mask": attention_mask
    }

    return X, labels

Then we can use `DataLoader` to create batches and shuffle the data for training.

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 256

train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)